# Ring Attention (环形注意力) 实现

Ring Attention是一种分布式注意力计算方法，通过环形通信突破单设备内存限制。

**核心思想**：
- 将序列分块到多个设备
- 使用环形通信模式传递K和V
- 累积计算完整的注意力输出
- 支持数百万token的序列

**公式**：
$$
\text{Attention}(Q_i) = \sum_{j=0}^{D-1} \text{softmax}(Q_i \cdot K_j^T) \cdot V_j
$$
通过$D$轮环形传递，在每个设备上计算完整的注意力。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def create_causal_mask(seq_len):
    """创建因果mask"""
    return np.tril(np.ones((seq_len, seq_len)))

## 2. 可视化环形通信模式

In [ ]:
# 可视化环形通信
num_devices = 4

fig, ax = plt.subplots(figsize=(10, 10))

# 设备位置（圆形排列）
angles = np.linspace(0, 2*np.pi, num_devices, endpoint=False)
radius = 3
positions = [(radius * np.cos(angle), radius * np.sin(angle)) 
             for angle in angles]

# 绘制设备
for i, (x, y) in enumerate(positions):
    circle = plt.Circle((x, y), 0.5, color='steelblue', alpha=0.7, ec='black', lw=2)
    ax.add_patch(circle)
    ax.text(x, y, f'设备{i}\n\nQ{i}\nK{i}, V{i}', 
            ha='center', va='center', fontsize=11, fontweight='bold')

# 绘制环形传递箭头
for i in range(num_devices):
    start = positions[i]
    end = positions[(i + 1) % num_devices]
    
    # 计算箭头的起点和终点（在圆的边缘）
    dx = end[0] - start[0]
    dy = end[1] - start[1]
    dist = np.sqrt(dx**2 + dy**2)
    
    start_x = start[0] + 0.5 * dx / dist
    start_y = start[1] + 0.5 * dy / dist
    end_x = end[0] - 0.5 * dx / dist
    end_y = end[1] - 0.5 * dy / dist
    
    arrow = FancyArrowPatch((start_x, start_y), (end_x, end_y),
                           arrowstyle='->', mutation_scale=30, lw=2,
                           color='red', alpha=0.7)
    ax.add_patch(arrow)
    
    # 添加传递的数据标签
    mid_x = (start_x + end_x) / 2
    mid_y = (start_y + end_y) / 2
    ax.text(mid_x, mid_y, f'K{i}, V{i}', fontsize=9, 
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Ring Attention 环形通信模式', fontsize=14, fontweight='bold', pad=20)

# 添加说明
ax.text(0, -4.5, '每轮K和V沿箭头方向传递到下一个设备\n每个设备保持自己的Q不变，累积计算注意力',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()

print("环形通信说明:")
print("  • 每个设备持有序列的一个块（Q_i, K_i, V_i）")
print("  • K和V按环形顺序传递")
print("  • 每轮每个设备计算：Q_i × K_j, V_j")
print(f"  • {num_devices}轮后完成所有计算")

## 3. 实现Ring Attention类

In [ ]:
class RingAttention:
    """
    Ring Attention (环形注意力)
    
    分布式计算注意力，突破单设备内存限制
    """
    
    def __init__(self, embed_dim, num_heads, num_devices=4, use_bias=True):
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.num_devices = num_devices
        self.use_bias = use_bias
        
        # Q, K, V投影
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
            self.b_o = np.zeros(embed_dim)
    
    def split_heads(self, x):
        """分割成多个头"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """合并多个头"""
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def split_sequence_to_devices(self, x):
        """将序列分割到多个设备"""
        seq_len = x.shape[0]
        chunk_size = seq_len // self.num_devices
        
        chunks = []
        for i in range(self.num_devices):
            start = i * chunk_size
            if i == self.num_devices - 1:
                end = seq_len
            else:
                end = (i + 1) * chunk_size
            chunks.append(x[start:end])
        
        return chunks
    
    def ring_attention_single_head(self, Q, K, V, mask=None):
        """计算单个头的环形注意力"""
        seq_len = Q.shape[0]
        
        # 分块到设备
        Q_chunks = self.split_sequence_to_devices(Q)
        K_chunks = self.split_sequence_to_devices(K)
        V_chunks = self.split_sequence_to_devices(V)
        
        # 初始化输出和归一化因子
        outputs = [np.zeros_like(Q_chunk) for Q_chunk in Q_chunks]
        max_scores = [np.full((Q_chunk.shape[0],), -np.inf)
                      for Q_chunk in Q_chunks]
        sum_exp = [np.zeros(Q_chunk.shape[0]) for Q_chunk in Q_chunks]
        
        attention_weights_full = np.zeros((seq_len, seq_len))
        
        # 环形传递
        for ring_step in range(self.num_devices):
            for device_id in range(self.num_devices):
                kv_source_device = (device_id + ring_step) % self.num_devices
                
                Q_i = Q_chunks[device_id]
                K_j = K_chunks[kv_source_device]
                V_j = V_chunks[kv_source_device]
                
                # 计算注意力得分
                scores = np.dot(Q_i, K_j.T) / np.sqrt(self.head_dim)
                
                # 应用mask
                if mask is not None:
                    q_start = sum([Q_chunks[d].shape[0] for d in range(device_id)])
                    k_start = sum([K_chunks[d].shape[0] for d in range(kv_source_device)])
                    q_end = q_start + Q_i.shape[0]
                    k_end = k_start + K_j.shape[0]
                    
                    mask_block = mask[q_start:q_end, k_start:k_end]
                    scores = np.where(mask_block == 0, -1e9, scores)
                
                # 在线Softmax
                current_max = np.max(scores, axis=1)
                new_max = np.maximum(max_scores[device_id], current_max)
                
                scale_old = np.exp(max_scores[device_id] - new_max)
                outputs[device_id] *= scale_old[:, None]
                sum_exp[device_id] *= scale_old
                
                scale_new = np.exp(current_max - new_max)
                exp_scores = np.exp(scores - current_max[:, None])
                
                outputs[device_id] += scale_new[:, None] * np.dot(exp_scores, V_j)
                sum_exp[device_id] += scale_new * np.sum(exp_scores, axis=1)
                
                max_scores[device_id] = new_max
                
                # 记录注意力权重
                attention_weights_full[q_start:q_end, k_start:k_end] = exp_scores
        
        # 最终归一化
        for device_id in range(self.num_devices):
            outputs[device_id] /= sum_exp[device_id][:, None]
        
        output = np.concatenate(outputs, axis=0)
        attention_weights = attention_weights_full / attention_weights_full.sum(axis=1, keepdims=True)
        
        return output, attention_weights
    
    def forward(self, query, key=None, value=None, mask=None, return_attention=False):
        """前向传播"""
        if key is None:
            key = query
        if value is None:
            value = key
        
        # 线性投影
        Q = np.dot(query, self.W_q)
        K = np.dot(key, self.W_k)
        V = np.dot(value, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 分割头
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 对每个头计算环形注意力
        head_outputs = []
        attention_weights_list = []
        
        for h in range(self.num_heads):
            head_output, attn_weights = self.ring_attention_single_head(
                Q[h], K[h], V[h], mask=mask
            )
            head_outputs.append(head_output)
            attention_weights_list.append(attn_weights)
        
        # 合并头
        multi_head_output = np.stack(head_outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        
        # 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            attention_weights = np.stack(attention_weights_list, axis=0)
            return output, attention_weights
        
        return output


class RingSelfAttention(RingAttention):
    """环形自注意力（简化接口）"""
    
    def forward(self, x, mask=None, return_attention=False):
        return super().forward(x, x, x, mask=mask, return_attention=return_attention)

## 4. 测试Ring Attention

In [ ]:
# 参数设置
seq_len = 16
embed_dim = 64
num_heads = 4
num_devices = 4

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  注意力头数: {num_heads}")
print(f"  设备数量: {num_devices}")
print(f"  每个设备的块大小: {seq_len // num_devices}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建Ring Attention层
ring_attn = RingSelfAttention(embed_dim, num_heads, num_devices)

# 前向传播
output, attn_weights = ring_attn.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")

## 5. 可视化设备分块

In [ ]:
# 可视化序列分块到设备
chunk_size = seq_len // num_devices

fig, ax = plt.subplots(figsize=(14, 4))

colors = ['lightcoral', 'lightblue', 'lightgreen', 'lightyellow']

for device_id in range(num_devices):
    start = device_id * chunk_size
    end = start + chunk_size
    
    # 绘制块
    for i in range(start, end):
        rect = plt.Rectangle((i, 0), 1, 1, 
                            facecolor=colors[device_id], 
                            edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(i + 0.5, 0.5, str(i), ha='center', va='center', fontsize=10)
    
    # 添加设备标签
    ax.text(start + chunk_size/2, -0.5, f'设备{device_id}', 
            ha='center', va='top', fontsize=12, fontweight='bold')

ax.set_xlim(0, seq_len)
ax.set_ylim(-1, 1.5)
ax.axis('off')
ax.set_title('序列分块到设备', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"设备分块（每个设备{chunk_size}个token）:")
for device_id in range(num_devices):
    start = device_id * chunk_size
    end = start + chunk_size
    print(f"  设备{device_id}: token [{start:2d}:{end:2d})")

## 6. 环形传递过程演示

In [ ]:
# 可视化环形传递的每一轮
print("环形传递过程:\n")
print(f"{'轮次':>6} | {'设备0':^15} | {'设备1':^15} | {'设备2':^15} | {'设备3':^15}")
print("-" * 80)

for ring_step in range(num_devices):
    row = f"{ring_step:>6} |"
    for device_id in range(num_devices):
        kv_source = (device_id + ring_step) % num_devices
        row += f" Q{device_id}×K{kv_source},V{kv_source} |"
    print(row)

print("\n说明:")
print("  • Q_i: 设备i的Query块（固定不变）")
print("  • K_j, V_j: 设备j的Key/Value块（环形传递）")
print("  • 每轮计算Q_i与K_j, V_j的注意力并累积")
print(f"  • {num_devices}轮后得到完整的注意力输出")

## 7. 数值验证：对比标准注意力

In [ ]:
# 标准注意力实现（作为对照）
def standard_attention(Q, K, V):
    """标准注意力"""
    scores = np.dot(Q, K.T) / np.sqrt(Q.shape[-1])
    attn_weights = softmax(scores, axis=-1)
    output = np.dot(attn_weights, V)
    return output, attn_weights

# 使用相同的Q, K, V
Q_test = x @ ring_attn.W_q
K_test = x @ ring_attn.W_k
V_test = x @ ring_attn.W_v

# 取第一个头进行对比
Q_test_h0 = ring_attn.split_heads(Q_test)[0]
K_test_h0 = ring_attn.split_heads(K_test)[0]
V_test_h0 = ring_attn.split_heads(V_test)[0]

# 标准注意力
output_standard, attn_standard = standard_attention(Q_test_h0, K_test_h0, V_test_h0)

# Ring Attention
output_ring, attn_ring = ring_attn.ring_attention_single_head(Q_test_h0, K_test_h0, V_test_h0)

# 计算差异
output_diff = np.abs(output_standard - output_ring).mean()
attn_diff = np.abs(attn_standard - attn_ring).mean()

print("数值验证（第1个头）:\n")
print(f"  标准注意力输出: {output_standard.shape}")
print(f"  Ring注意力输出: {output_ring.shape}")
print(f"\n  输出平均绝对差异: {output_diff:.6f}")
print(f"  注意力权重平均绝对差异: {attn_diff:.6f}")

if output_diff < 1e-5:
    print("\n  ✓ Ring Attention数值正确（与标准注意力一致）")
else:
    print("\n  ⚠ 存在数值差异（可能由于浮点精度）")

# 可视化对比
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 标准注意力
sns.heatmap(attn_standard, annot=False, cmap='YlOrRd',
            ax=ax1, cbar=True, square=True)
ax1.set_title('标准注意力', fontsize=12, fontweight='bold')
ax1.set_xlabel('Key位置')
ax1.set_ylabel('Query位置')

# Ring注意力
sns.heatmap(attn_ring, annot=False, cmap='YlOrRd',
            ax=ax2, cbar=True, square=True)
ax2.set_title('Ring Attention', fontsize=12, fontweight='bold')
ax2.set_xlabel('Key位置')
ax2.set_ylabel('Query位置')

# 差异
diff_map = np.abs(attn_standard - attn_ring)
sns.heatmap(diff_map, annot=False, cmap='Reds',
            ax=ax3, cbar=True, square=True)
ax3.set_title(f'绝对差异 (平均={attn_diff:.6f})', fontsize=12, fontweight='bold')
ax3.set_xlabel('Key位置')
ax3.set_ylabel('Query位置')

plt.tight_layout()
plt.show()

## 8. 内存和通信分析

In [ ]:
def analyze_memory_communication(seq_len, num_devices, embed_dim):
    """分析内存和通信"""
    chunk_size = seq_len // num_devices
    
    # 单设备标准注意力内存
    single_attn_mem = seq_len * seq_len * 4  # 注意力矩阵（float32）
    single_kv_mem = 2 * seq_len * embed_dim * 4  # K和V
    single_total = single_attn_mem + single_kv_mem
    
    # Ring Attention每个设备内存
    ring_attn_mem = chunk_size * chunk_size * 4
    ring_kv_mem = 2 * chunk_size * embed_dim * 4
    ring_total = ring_attn_mem + ring_kv_mem
    
    # 通信量
    kv_transfer_per_step = 2 * chunk_size * embed_dim * 4
    total_communication = kv_transfer_per_step * (num_devices - 1)
    
    return {
        'single_total_mb': single_total / 1024 / 1024,
        'ring_total_mb': ring_total / 1024 / 1024,
        'memory_reduction': num_devices,
        'total_comm_mb': total_communication / 1024 / 1024
    }

# 分析不同配置
configs = [
    (10_000, 4, 512),
    (100_000, 8, 1024),
    (1_000_000, 16, 2048),
]

print("Ring Attention 内存和通信分析:\n")
print(f"{'序列长度':>12} {'设备数':>8} {'单设备(MB)':>12} {'Ring/设备(MB)':>15} {'内存减少':>10} {'通信量(MB)':>12}")
print("-" * 85)

for seq_len, num_dev, emb_dim in configs:
    stats = analyze_memory_communication(seq_len, num_dev, emb_dim)
    print(f"{seq_len:>12,} {num_dev:>8} {stats['single_total_mb']:>12.1f} "
          f"{stats['ring_total_mb']:>15.1f} {stats['memory_reduction']:>9.0f}x "
          f"{stats['total_comm_mb']:>12.1f}")

print("\n关键观察:")
print("  ✓ 内存减少倍数 = 设备数")
print("  ✓ 100万token序列在16设备上可行")
print("  ✓ 通信量相对较小（可与计算重叠）")

## 9. 可扩展性分析

In [ ]:
# 不同设备数下的序列长度上限
single_device_memory_mb = 24000  # 24GB显存
embed_dim_test = 4096

device_counts = [1, 2, 4, 8, 16, 32, 64]
max_seq_lens = []

for num_dev in device_counts:
    # 估算单设备能支持的块大小
    # 主要限制: chunk_size^2 * 4 bytes (注意力矩阵)
    chunk_size = int(np.sqrt(single_device_memory_mb * 1024 * 1024 / 4))
    total_seq_len = chunk_size * num_dev
    max_seq_lens.append(total_seq_len)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：序列长度 vs 设备数
ax1.plot(device_counts, max_seq_lens, marker='o', linewidth=2, markersize=10)
ax1.set_xlabel('设备数量', fontsize=12)
ax1.set_ylabel('最大序列长度', fontsize=12)
ax1.set_title('Ring Attention 可扩展性', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)
ax1.set_yscale('log')

# 添加数值标签
for x, y in zip(device_counts, max_seq_lens):
    if y >= 1_000_000:
        label = f'{y/1_000_000:.1f}M'
    elif y >= 1000:
        label = f'{y/1000:.0f}K'
    else:
        label = f'{y}'
    ax1.text(x, y, label, ha='center', va='bottom', fontsize=9)

# 右图：每个设备的内存占用
memory_per_device = [single_device_memory_mb / num_dev for num_dev in device_counts]
ax2.bar(range(len(device_counts)), memory_per_device, color='steelblue', alpha=0.7)
ax2.set_xlabel('设备数量', fontsize=12)
ax2.set_ylabel('每设备内存 (MB)', fontsize=12)
ax2.set_title('内存分布', fontsize=13, fontweight='bold')
ax2.set_xticks(range(len(device_counts)))
ax2.set_xticklabels(device_counts)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 打印详细数据
print("\n扩展性分析（单设备24GB显存）:\n")
print(f"{'设备数':>8} {'最大序列长度':>15} {'每设备内存(GB)':>18}")
print("-" * 45)

for num_dev, seq_len in zip(device_counts, max_seq_lens):
    mem_gb = single_device_memory_mb / num_dev / 1024
    print(f"{num_dev:>8} {seq_len:>15,} {mem_gb:>18.1f}")

print("\n关键结论:")
print("  ✓ 设备数翻倍，支持的序列长度翻倍")
print("  ✓ 64个设备可以处理数百万token")
print("  ✓ 线性可扩展性")

## 10. 实际应用场景

In [ ]:
# 实际应用场景示例
applications = [
    {'name': '长文档理解', 'seq_len': 100_000, 'devices': 8},
    {'name': '全书分析', 'seq_len': 500_000, 'devices': 32},
    {'name': '长视频理解', 'seq_len': 1_000_000, 'devices': 64},
    {'name': '基因组序列', 'seq_len': 3_000_000, 'devices': 128},
]

print("Ring Attention 实际应用场景:\n")
print(f"{'应用':^15} {'序列长度':>12} {'所需设备':>10} {'每设备块大小':>15}")
print("-" * 60)

for app in applications:
    chunk = app['seq_len'] // app['devices']
    print(f"{app['name']:^15} {app['seq_len']:>12,} {app['devices']:>10} {chunk:>15,}")

print("\n\nRing Attention的优势:")
print("  ✓ 突破单设备内存限制")
print("  ✓ 线性扩展性（设备数）")
print("  ✓ 计算与通信可重叠")
print("  ✓ 保持完整注意力（非近似）")
print("  ✓ 与Flash Attention等优化兼容")

print("\n应用领域:")
print("  • 超长文档处理（法律文书、学术论文）")
print("  • 全书级别文本分析")
print("  • 长视频理解（数小时视频）")
print("  • 基因组序列分析")
print("  • 代码库级别的代码理解")
print("  • 极长上下文对话系统")

## 总结

### Ring Attention的核心要点：

1. **核心机制**：
   - 序列分块到多个设备
   - K和V在设备间环形传递
   - 每个设备累积计算注意力输出
   - D轮后得到完整结果

2. **关键优势**：
   - 突破单设备内存限制
   - 线性可扩展性
   - 支持数百万token序列
   - 完整注意力计算（非近似）

3. **实现要点**：
   - 在线Softmax算法（数值稳定）
   - 环形通信模式
   - 计算与通信重叠
   - 兼容现有优化技术

4. **应用场景**：
   - 超长文档处理
   - 长视频理解
   - 基因组分析
   - 需要极长上下文的任务

### 复杂度分析：

**内存**：
$$
\text{每设备内存} = O\left(\frac{n^2}{D^2} + \frac{n \cdot d}{D}\right)
$$

**通信**：
$$
\text{总通信量} = O\left(\frac{n \cdot d}{D} \cdot D\right) = O(n \cdot d)
$$

**关键**：内存减少$D$倍，通信量与序列长度线性相关！